# Phase 5 — SQL Analytics

## 5.1 Objective

The dimensional model is now ready for analysis.

In this phase, SQL will be used to turn the modeled sales data into business insights across sales performance, customers, products, categories, and time.

The focus is not just on writing queries, but on answering questions that can support the later KPI, dashboard, and root-cause analysis stages.

In [71]:
import pandas as pd

## 5.2 Connect to the Analytical Data

The four validated analytical tables created in Phase 4 will be loaded into a SQL environment.

Before starting the analysis, we will confirm that the tables and relationships are available as expected.

In [72]:
import mysql.connector

In [73]:
print(mysql.connector.__version__)

26.7.0


In [74]:
import sys
print(sys.executable)

c:\Users\ASUS\Desktop\Data science\lol\project 2\Retail-Revenue-Intelligence\.venv\Scripts\python.exe


In [75]:
import os
from dotenv import load_dotenv

load_dotenv()

conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

print("Connected successfully!")

Connected successfully!


### 5.2.1 Load the Analytical Tables

The validated tables from Phase 4 will now be loaded into MySQL:

- `dim_date`
- `dim_customer`
- `dim_product`
- `fact_sales`

The SQL layer will use these tables as the foundation for the analysis that follows.

### 5.3.1 Create Table Structures

The MySQL tables will mirror the final star-schema design from Phase 4, including the surrogate keys and relationships between the dimensions and fact table.

In [76]:
from pathlib import Path

print(Path.cwd())

c:\Users\ASUS\Desktop\Data science\lol\project 2\Retail-Revenue-Intelligence\notebooks


In [77]:
cursor = conn.cursor()

cursor.execute("SELECT DATABASE()")
print(cursor.fetchone()[0])

retail_analytics


In [78]:
cursor.execute("SHOW TABLES")
tables = cursor.fetchall()

tables

[('dim_customer',), ('dim_date',), ('dim_product',), ('fact_sales',)]

### dim_date table

In [42]:
cursor.execute("""
CREATE TABLE dim_date (
    date_key INT PRIMARY KEY,
    date DATE NOT NULL,
    year INT,
    quarter INT,
    month INT,
    month_name VARCHAR(20),
    day INT,
    day_name VARCHAR(20),
    week INT,
    is_weekend BOOLEAN
)
""")

conn.commit()

print("dim_date recreated successfully!")

ProgrammingError: 1050 (42S01): Table 'dim_date' already exists

In [ ]:
cursor.execute("SHOW TABLES")
cursor.fetchall()

[('dim_customer',), ('dim_date',), ('dim_product',), ('fact_sales',)]

### dim_customer table

In [ ]:
cursor.execute("""
CREATE TABLE dim_customer (
    customer_key INT PRIMARY KEY,
    customer_id INT NOT NULL
)
""")

conn.commit()

print("dim_customer table created successfully!")

ProgrammingError: 1050 (42S01): Table 'dim_customer' already exists

In [ ]:
cursor.execute("SHOW TABLES")
cursor.fetchall()

[('dim_customer',), ('dim_date',), ('dim_product',), ('fact_sales',)]

### dim_product table

In [ ]:
cursor.execute("""
CREATE TABLE dim_product (
    product_key INT PRIMARY KEY,
    sku VARCHAR(50) NOT NULL,
    sku_category VARCHAR(255)
)
""")

conn.commit()

print("dim_product table created successfully!")

dim_product table created successfully!


In [ ]:
cursor.execute("SHOW TABLES")
cursor.fetchall()

[('dim_customer',), ('dim_date',), ('dim_product',), ('fact_sales',)]

### fact_sales table

In [80]:
cursor.execute("""
CREATE TABLE fact_sales (
    transaction_id INT NOT NULL,
    date_key INT NOT NULL,
    customer_key INT NOT NULL,
    product_key INT NOT NULL,
    quantity DECIMAL(12,3) NOT NULL,
    sales_amount DECIMAL(12,2) NOT NULL,

    PRIMARY KEY (transaction_id, product_key),

    FOREIGN KEY (date_key)
        REFERENCES dim_date(date_key),

    FOREIGN KEY (customer_key)
        REFERENCES dim_customer(customer_key),

    FOREIGN KEY (product_key)
        REFERENCES dim_product(product_key)
)
""")

conn.commit()

print("fact_sales recreated successfully!")

fact_sales recreated successfully!


In [49]:
cursor.execute("SHOW TABLES")
cursor.fetchall()

[('dim_customer',), ('dim_date',), ('dim_product',), ('fact_sales',)]

### 5.3.5 Load the CSV Data

The validated dimensional and fact datasets created in Phase 4 are now loaded into MySQL. The dimension tables are loaded first so that their keys are available when the fact table is populated.

In [50]:
output_path = Path("../data/processed")

In [51]:
list(output_path.glob("*.csv"))

[WindowsPath('../data/processed/dim_customer.csv'),
 WindowsPath('../data/processed/dim_date.csv'),
 WindowsPath('../data/processed/dim_product.csv'),
 WindowsPath('../data/processed/fact_sales.csv'),
 WindowsPath('../data/processed/scanner_data_clean.csv')]

### Load dim_date

In [52]:
dim_date_df = pd.read_csv(output_path / "dim_date.csv")

dim_date_df.head()

,date_key,date,year,quarter,month,month_name,day,day_name,week,is_weekend
0,1,2016-01-02,2016,1,1,January,2,Saturday,53,True
1,2,2016-01-03,2016,1,1,January,3,Sunday,53,True
2,3,2016-01-04,2016,1,1,January,4,Monday,1,False
3,4,2016-01-05,2016,1,1,January,5,Tuesday,1,False
4,5,2016-01-06,2016,1,1,January,6,Wednesday,1,False


In [ ]:
for row in dim_date_df.itertuples(index=False, name=None):
    cursor.execute(
        """
        INSERT INTO dim_date
        (date_key, date, year, quarter, month, month_name,
         day, day_name, week, is_weekend)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """,
        row
    )

conn.commit()

print(f"Loaded {len(dim_date_df):,} rows into dim_date.")

IntegrityError: 1062 (23000): Duplicate entry '1' for key 'dim_date.PRIMARY'

In [ ]:
cursor.execute("SELECT COUNT(*) FROM dim_date")
print(cursor.fetchone()[0])

363


### Load dim_customer

In [ ]:
dim_customer_df = pd.read_csv(output_path / "dim_customer.csv")

dim_customer_df.head()

,customer_key,customer_id
0,1,1
1,2,2
2,3,3
3,4,4
4,5,5


In [ ]:
for row in dim_customer_df.itertuples(index=False, name=None):
    cursor.execute(
        """
        INSERT INTO dim_customer
        (customer_key, customer_id)
        VALUES (%s, %s)
        """,
        row
    )

conn.commit()

print(f"Loaded {len(dim_customer_df):,} rows into dim_customer.")

Loaded 22,625 rows into dim_customer.


In [ ]:
cursor.execute("SELECT COUNT(*) FROM dim_customer")
print(cursor.fetchone()[0])

22625


### Load dim_product

In [ ]:
dim_product_df = pd.read_csv(output_path / "dim_product.csv")

dim_product_df.head()

,product_key,sku,sku_category
0,1,00GVC,1TS
1,2,00OK1,J4R
2,3,0121I,XG4
3,4,01IEO,XVK
4,5,01IQT,Z4O


In [ ]:
for row in dim_product_df.itertuples(index=False, name=None):
    cursor.execute(
        """
        INSERT INTO dim_product
        (product_key, sku, sku_category)
        VALUES (%s, %s, %s)
        """,
        row
    )

conn.commit()

print(f"Loaded {len(dim_product_df):,} rows into dim_product.")

Loaded 5,242 rows into dim_product.


In [ ]:
cursor.execute("SELECT COUNT(*) FROM dim_product")
print(cursor.fetchone()[0])

5242


### Load fact_sales

In [81]:
fact_sales_df = pd.read_csv(output_path / "fact_sales.csv")

fact_sales_df.head()

,transaction_id,date_key,customer_key,product_key,quantity,sales_amount
0,1,1,2547,64,1.0,3.13
1,2,1,822,923,1.0,5.46
2,3,1,3686,1933,1.0,6.35
3,4,1,3719,777,1.0,5.59
4,5,1,9200,2972,1.0,6.88


In [82]:
fact_sales_df.columns.tolist()

['transaction_id',
 'date_key',
 'customer_key',
 'product_key',
 'quantity',
 'sales_amount']

In [83]:
print("Rows:", len(fact_sales_df))

print("Zero quantities:",
      (fact_sales_df['quantity'] == 0).sum())

print("Negative quantities:",
      (fact_sales_df['quantity'] < 0).sum())

print("NULL quantities:",
      fact_sales_df['quantity'].isna().sum())

print("Total quantity:",
      fact_sales_df['quantity'].sum())

Rows: 131706
Zero quantities: 0
Negative quantities: 0
NULL quantities: 0
Total quantity: 195624.429


In [84]:
fact_sales_rows = list(
    fact_sales_df.itertuples(index=False, name=None)
)

insert_query = """
INSERT INTO fact_sales
(transaction_id, date_key, customer_key, product_key, quantity, sales_amount)
VALUES (%s, %s, %s, %s, %s, %s)
"""

batch_size = 5000

for i in range(0, len(fact_sales_rows), batch_size):
    batch = fact_sales_rows[i:i + batch_size]
    cursor.executemany(insert_query, batch)
    conn.commit()

print(f"Loaded {len(fact_sales_df):,} rows into fact_sales.")

Loaded 131,706 rows into fact_sales.


In [85]:
query = "SELECT * FROM fact_sales LIMIT 10"

fact_sales_check = pd.read_sql(query, conn)

fact_sales_check

C:\Users\ASUS\AppData\Local\Temp\ipykernel_18708\1709205366.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fact_sales_check = pd.read_sql(query, conn)


,transaction_id,date_key,customer_key,product_key,quantity,sales_amount
0,1,1,2547,64,1.0,3.13
1,2,1,822,923,1.0,5.46
2,3,1,3686,1933,1.0,6.35
3,4,1,3719,777,1.0,5.59
4,5,1,9200,2972,1.0,6.88
5,6,1,5010,2474,1.0,10.77
6,7,1,1666,1553,1.0,3.65
7,7,1,1666,1557,1.0,8.21
8,8,1,1253,1457,1.0,8.25
9,9,1,5541,1122,1.0,8.18


In [86]:
cursor.execute("SELECT COUNT(*) FROM fact_sales")

fact_sales_count = cursor.fetchone()[0]

print(f"MySQL fact_sales rows: {fact_sales_count:,}")

MySQL fact_sales rows: 131,706


In [87]:
cursor.execute("DESCRIBE fact_sales")

cursor.fetchall()

[('transaction_id', 'int', 'NO', 'PRI', None, ''),
 ('date_key', 'int', 'NO', 'MUL', None, ''),
 ('customer_key', 'int', 'NO', 'MUL', None, ''),
 ('product_key', 'int', 'NO', 'PRI', None, ''),
 ('quantity', 'decimal(12,3)', 'NO', '', None, ''),
 ('sales_amount', 'decimal(12,2)', 'NO', '', None, '')]

In [88]:
print(f"Source fact_sales rows: {len(fact_sales_df):,}")
print(f"Match: {fact_sales_count == len(fact_sales_df)}")

Source fact_sales rows: 131,706
Match: True


Validate All Table Row Counts

In [89]:
expected_counts = {
    "dim_date": 363,
    "dim_customer": 22625,
    "dim_product": 5242,
    "fact_sales": 131706
}

for table, expected in expected_counts.items():
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    actual = cursor.fetchone()[0]

    print(
        f"{table:<15} "
        f"Expected: {expected:>7,} | "
        f"Actual: {actual:>7,} | "
        f"Match: {actual == expected}"
    )

dim_date        Expected:     363 | Actual:     363 | Match: True
dim_customer    Expected:  22,625 | Actual:  22,625 | Match: True
dim_product     Expected:   5,242 | Actual:   5,242 | Match: True
fact_sales      Expected: 131,706 | Actual: 131,706 | Match: True


Referential Integrity Check

In [90]:
ri_checks = {
    "date_key": """
        SELECT COUNT(*)
        FROM fact_sales f
        LEFT JOIN dim_date d
            ON f.date_key = d.date_key
        WHERE d.date_key IS NULL
    """,

    "customer_key": """
        SELECT COUNT(*)
        FROM fact_sales f
        LEFT JOIN dim_customer c
            ON f.customer_key = c.customer_key
        WHERE c.customer_key IS NULL
    """,

    "product_key": """
        SELECT COUNT(*)
        FROM fact_sales f
        LEFT JOIN dim_product p
            ON f.product_key = p.product_key
        WHERE p.product_key IS NULL
    """
}

for key, query in ri_checks.items():
    cursor.execute(query)
    orphan_rows = cursor.fetchone()[0]

    print(f"{key:<15} Orphan rows: {orphan_rows}")

date_key        Orphan rows: 0
customer_key    Orphan rows: 0
product_key     Orphan rows: 0


Validate Primary Keys

In [91]:
pk_checks = {
    "dim_date": "date_key",
    "dim_customer": "customer_key",
    "dim_product": "product_key"
}

for table, key in pk_checks.items():
    cursor.execute(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT {key}) AS unique_keys,
            COUNT({key}) AS non_null_keys
        FROM {table}
    """)

    total, unique, non_null = cursor.fetchone()

    print(f"\n{table}")
    print(f"Total rows:    {total:,}")
    print(f"Unique keys:   {unique:,}")
    print(f"Non-null keys: {non_null:,}")
    print(f"Valid: {total == unique == non_null}")


dim_date
Total rows:    363
Unique keys:   363
Non-null keys: 363
Valid: True

dim_customer
Total rows:    22,625
Unique keys:   22,625
Non-null keys: 22,625
Valid: True

dim_product
Total rows:    5,242
Unique keys:   5,242
Non-null keys: 5,242
Valid: True


In [92]:
cursor.execute("""
    SELECT
        TABLE_NAME,
        COLUMN_NAME,
        REFERENCED_TABLE_NAME,
        REFERENCED_COLUMN_NAME
    FROM information_schema.KEY_COLUMN_USAGE
    WHERE TABLE_SCHEMA = DATABASE()
      AND TABLE_NAME = 'fact_sales'
      AND REFERENCED_TABLE_NAME IS NOT NULL
""")

for row in cursor.fetchall():
    print(row)

('fact_sales', 'date_key', 'dim_date', 'date_key')
('fact_sales', 'customer_key', 'dim_customer', 'customer_key')
('fact_sales', 'product_key', 'dim_product', 'product_key')


## 5.4 SQL Analytics Layer

The MySQL star schema is now loaded and validated. Analytical queries are maintained separately as SQL scripts to keep the database analysis modular, reproducible, and easy to review.